In [1]:
from pathlib import Path

import pandas as pd

from QuantNado import BamStore
from QuantNado.downstream import (
    annotate_intervals,
    extract_feature_ranges,
    extract_metadata,
    extract_promoters,
    load_gtf,
    plot_pca_scatter,
    plot_pca_scree,
    reduce_byranges_signal,
    run_pca,
)


In [2]:
fig_dir = Path("./figures")
fig_dir.mkdir(exist_ok=True)

# Load Dataset

In [3]:
ds = BamStore.open("dataset", backend="zarr")
ds

2025-12-20 00:06:27.701 | INFO     | QuantNado.bam_store:open:372 - Opening zarr store at: dataset.zarr


<xarray.Dataset> Size: 4GB
Dimensions:        (sample: 8, position_flat: 154755866, contig: 3)
Coordinates:
  * sample         (sample) int64 64B 0 1 2 3 4 5 6 7
  * position_flat  (position_flat) int64 1GB 0 1 2 ... 154755864 154755865
  * contig         (contig) int64 24B 0 1 2
    contig_length  (contig) int64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    contig_offset  (contig) int64 24B dask.array<chunksize=(3,), meta=np.ndarray>
Data variables:
    signal         (sample, position_flat) uint16 2GB dask.array<chunksize=(1, 64000), meta=np.ndarray>
Attributes: (12/21)
    assay_by_sample:         ['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP',...
    metadata_control:        ['', '', 'Input', 'Input', 'Input', 'Input', '',...
    metadata_ip:             ['', '', 'MLL', 'MLL', 'MLL', 'MLL', '', '']
    metadata_replicate:      ['', '', '', '', '', '', 'rep1', 'rep1']
    metadata_scaling_group:  ['default', 'default', 'default', 'default', 'de...
    metadata_timepoint:      ['24hr', '24hr', '24hr', '24hr', '24hr', '24hr',...
    ...                      ...
    assays:                  ATAC,ChIP,RNA
    sample_names:            ['SEM-DMSO', 'SEM-MENi', 'SEM-DMSO-MLL_Input', '...
    contig_names:            ['chr21', 'chr22', 'chrY']
    structure:               ragged (sample × position_flat with contig offsets)
    bin_size:                1
    average_sparsity:        85.64%

# Explore Zarr Dataset



In [4]:
# Check dimensions and coordinates
print("Dimensions:", ds.dims)
print("\nCoordinates:", list(ds.coords))
print("\nAttributes:", ds.attrs)
print("\nData variables:", list(ds.data_vars))

Dimensions: FrozenMappingWarningOnValuesAccess({'sample': 8, 'position_flat': 154755866, 'contig': 3})

Coordinates: ['contig', 'position_flat', 'contig_length', 'contig_offset', 'sample']

Attributes: {'assay_by_sample': ['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP', 'RNA', 'RNA'], 'metadata_control': ['', '', 'Input', 'Input', 'Input', 'Input', '', ''], 'metadata_ip': ['', '', 'MLL', 'MLL', 'MLL', 'MLL', '', ''], 'metadata_replicate': ['', '', '', '', '', '', 'rep1', 'rep1'], 'metadata_scaling_group': ['default', 'default', 'default', 'default', 'default', 'default', 'default', 'default'], 'metadata_timepoint': ['24hr', '24hr', '24hr', '24hr', '24hr', '24hr', '24hr', '24hr'], 'metadata_treatment': ['DMSO', 'MENi', 'DMSO', 'DMSO', 'MENi', 'MENi', 'DMSO', 'MENi'], 'metadata_r1': ['/ceph/project/milne_group/cchahrou/project/2025-12-17_menin_inh_24hr/fastqs/atac/SEM-DMSO_R1.fastq.gz', '/ceph/project/milne_group/cchahrou/project/2025-12-17_menin_inh_24hr/fastqs/atac/SEM-MENi_R1.fastq.gz

In [5]:
# Helpers for ragged coordinates and metadata
metadata_df = extract_metadata(ds)

# Reduce by bed file

In [ ]:
promoters = "/Users/catherine/work/project/QuantNado/data/hg38/promoters_1024bp.bed"

promoter_ds = reduce_byranges_signal(ds["signal"], bed_file=promoters, reduction="mean")
promoter_ds

<xarray.Dataset> Size: 27MB
Dimensions:       (ranges: 116978, sample: 8)
Coordinates:
  * ranges        (ranges) int64 936kB 0 1 2 3 4 ... 116974 116975 116976 116977
  * sample        (sample) int64 64B 0 1 2 3 4 5 6 7
    start         (ranges) int64 936kB 64907 381723 ... 154751054 154750565
    end           (ranges) int64 936kB 65931 382747 ... 154752078 154751589
    range_length  (ranges) int64 936kB 1024 1024 1024 1024 ... 1024 1024 1024
    contig        (ranges) object 936kB 'chr1' 'chr1' 'chr1' ... 'chrX' 'chrX'
Data variables:
    sum           (ranges, sample) uint64 7MB dask.array<chunksize=(116978, 1), meta=np.ndarray>
    count         (ranges, sample) int64 7MB dask.array<chunksize=(116978, 1), meta=np.ndarray>
    mean          (ranges, sample) float64 7MB dask.array<chunksize=(116978, 1), meta=np.ndarray>

## PCA analysis

Perform Principal Component Analysis on the promoter dataset to visualize sample relationships:

In [ ]:
# Ensure numeric dtype and explicit chunking to avoid auto on object dtypes
promoter_mean = promoter_ds["mean"].astype("float32")
promoter_signal = promoter_mean.chunk({"ranges": 10000})

pca_object, pca_result = run_pca(
    input_array=promoter_signal,
    n_components=2,
    nan_handling_strategy="drop",
    standardize=True,
    random_state=42,
    subset_size=5_000,
    subset_strategy="random",
    svd_solver="randomized",
)

plot_pca_scree(
    pca_object=pca_object,
    filepath=f"{fig_dir}/pca_scree.png",
)

plot_pca_scatter(
    pca_object,
    pca_result,
    xaxis_pc=1,
    yaxis_pc=2,
    metadata_df=metadata_df,
    colour_by="assay",
    shape_by="treatment",
    sample_column="sample_id",
    filepath=f"{fig_dir}/pca_plot.png",
)

# Extract GTF features

In [ ]:
gtf_file = "/Users/catherine/work/project/QuantNado/data/hg38/hg38.ncbiRefSeq.gtf"

gtf = load_gtf(gtf_file, feature_types=["gene", "transcript"])
gtf

In [ ]:
genes = extract_feature_ranges(gtf, feature_type="gene")
genes

In [ ]:
promoters_extracted = extract_promoters(
    gtf, 
    upstream=1000, 
    downstream=200, 
    anchor_feature="gene"
)

# Annotate reduced promoter windows (from BED) against gene promoters
promoter_peaks = pd.DataFrame(
    {
        "contig": promoter_ds["contig"].values
        if "contig" in promoter_ds.coords
        else "chr1",
        "start": promoter_ds["start"].values,
        "end": promoter_ds["end"].values,
    }
)

annotated_promoters = annotate_intervals(
    intervals=promoter_peaks,
    feature_df=promoters_extracted,
    feature_prefix="promoter_",
    require_overlap=False,
)
annotated_promoters.head()